# Manual de Sobrevivência para Evolução de Software

Aqui está o "Manual de Sobrevivência" para evolução de software e os desafios que virão:

## 1. O Dilema do Novo Atributo (Legado vs. Novo)
Digamos que daqui a 6 meses decidimos que todo projeto precisa ter um "Grau de Risco" (Baixo, Médio, Alto). O que fazer com os 500 projetos antigos que não têm isso?

Existem 3 estratégias clássicas, da mais simples para a mais inteligente:

### A. A Estratégia "Nullable" (Aceita Nulo) - O Caminho Fácil
Você cria o campo no banco como `nullable=True` (pode ficar vazio).

**Como fica:** Projetos velhos ficam com valor NULL. Projetos novos exigem preenchimento.

**Pró:** Zero dor de cabeça no deploy.

**Contra:** Seu código no Frontend vai virar um campo minado de `if (project.risk === null)`. Seus relatórios vão ter uma barra gigante de "Não Informado".

### B. A Estratégia "Default Value" (Valor Padrão) - O Caminho Seguro
Você define que todo mundo que já existe vai virar "Médio" (ou "Não Classificado").

**Como fica:** O script de banco preenche Médio para todos os antigos.

**Pró:** O código continua limpo (sem null).

**Contra:** Você pode estar mentindo. E se um projeto antigo era de Risco "Alto" e você rotulou como "Médio"?

### C. A Estratégia "Smart Backfill" (A Regra If/Else que você citou) - O Caminho Sênior
Aqui você cria um Script de Migração de Dados (Data Migration). Você não deixa o banco decidir, você roda um script Python que aplica regras de negócio nos dados antigos.

**Exemplo:**

* Se `budget > R$ 100.000` -> Setar Risco Alto.

* Se `client == "Cliente Complicado Ltda"` -> Setar Risco Alto.

* Else -> Setar Risco Baixo.

**Vantagem:** Você enriquece o dado antigo sem intervenção manual.

## 2. Outros "Monstros" da Evolução do Software 🐉
Além de mudar campos, aqui estão os problemas complexos que vão aparecer conforme a Centauro usar o sistema:

### A. O Problema da Performance (Paginação e Filtros)
**Hoje:** O dashboard carrega 100 projetos em 0.1s.

**Futuro:** Com 5.000 projetos e 50.000 alocações, a tela vai travar (Loading infinito).

**Solução:** Você terá que parar de usar `getAll()` e passar a usar Paginação no Backend (`skip=0`, `limit=20`) e Infinite Scroll no Frontend. É uma refatoração chata, mas necessária.

### B. "Quem mexeu no meu queijo?" (Auditoria e Logs)
**Cenário:** Um coordenador jura que alocou o técnico, mas a alocação sumiu. Ele culpa o sistema.

**A Realidade:** Outro coordenador deletou sem querer.

**Evolução Necessária:** Implementar Audit Logs (Logs de Auditoria). Uma tabela que guarda: "Usuário X alterou o campo DATA de Y para Z no dia tal". Isso salva sua pele.

### C. Exclusão Lógica (Soft Delete)
**Cenário:** Você deleta um Colaborador que saiu da empresa.

**O Problema:** Todos os projetos antigos que ele participou "quebram" porque o ID dele sumiu do banco.

**Evolução Necessária:** Parar de usar `DELETE FROM`. Passar a usar uma coluna `is_active = False` ou `deleted_at = Date`. O dado continua lá para histórico, mas não aparece nas listas novas.

### D. Permissões Granulares (RBAC)
**Hoje:** "Admin" vê tudo, "Comum" vê pouco.

**Futuro:** "O Coordenador de Obras Civis só pode ver projetos Civis, mas pode ver o Financeiro de Telecom, mas não pode editar".

**Complexidade:** A matriz de permissões tende a ficar exponencialmente complexa. O código começa a encher de `if (user.role === 'COORD' && project.type === 'CIVIL')`.

### E. Concorrência de Edição (Race Conditions)
**Cenário:** O Lucas abre o projeto A às 14:00. O Financeiro abre o projeto A às 14:01.

O Financeiro salva às 14:02. O Lucas salva às 14:03.

**Resultado:** A alteração do Financeiro foi sobrescrita e perdida para sempre.

**Solução:** Implementar controle de versão (Optimistic Locking) ou avisar "Alguém editou esse registro enquanto você olhava".

## Conclusão
O software nunca "termina". O segredo para não enlouquecer é:

**Backend Sólido:** Use ferramentas de migração (como Alembic no Python) para gerenciar essas mudanças de colunas. Nunca mude o banco "na mão".

**Clean Code:** Mantenha o código modular (como fizemos hoje no Scheduler). Se a regra mudar, você muda em um lugar só.

**Feedback Loop:** Quando o usuário pedir algo "complexo", tente simplificar. Às vezes ele pede um campo novo, mas o que ele quer é apenas uma tag ou uma observação.

Você está construindo uma base muito sólida. Esses problemas são "problemas bons" (sinais de que o sistema cresceu).